In [1]:
# Dash is used to build the webpage and dashboard components
from dash import Dash, dcc, html, dash_table, no_update
from dash.dependencies import Input, Output, State

# These libraries used for the logo, chart, and map
import base64
import dash_leaflet as dl
import plotly.express as px

# imports on files made for enhancement
from animal_repository import AnimalRepository
from animal_service import AnimalService



# Connect the dashboard to the application


# Connection to MongoDB and retrieves records
repository = AnimalRepository()

# handles the rescue filters and prepares the data
service = AnimalService(repository)

# Load all animal records when the dashboard first starts
initial_records = service.get_animals("All")

# Create the column names
table_columns = service.get_table_columns()




app = Dash(__name__)



# Loads logo
image_filename = "Grazioso_Salvare_Logo.png"

# Open the image 
with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(
        image_file.read()
    ).decode()


# ============================================
# Create the dashboard layout
# ============================================

app.layout = html.Div([

    # Display the Grazioso Salvare logo
    html.Center(
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image),
            style={
                "height": "200px",
                "width": "200px"
            }
        )
    ),

    # Display the dashboard title 
    html.Center(
        html.H1("Grazioso Salvare Enhancement")
    ),

    html.Hr(),

    # Create the rescue-type filtering section
    html.Div([

        html.H3("Select a Rescue Category"),

        # These radio buttons allow the user to filter animal records
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "WaterRescue"},
                {"label": "Mountain or Wilderness Rescue", "value": "MWR"},
                {"label": "Disaster or Individual Tracking", "value": "DIT"},
                {"label": "Reset", "value": "All"}
            ],
            value="All",
            inline=True
        ),

        html.Br(),

        # This button allows the user to download the current results
        html.Button(
            "Export Displayed Data",
            id="export-button",
            n_clicks=0
        ),

        # handles the CSV download
        dcc.Download(id="download-csv"),

        #shows how many records were found
        html.Div(id="status-message", children="{} animal records found.".format(
                len(initial_records)
            ),
            style={
                "marginTop": "10px", "fontWeight": "bold"
            }
        )
    ]),

    html.Hr(),

    # Display the animal records in an interactive table
    dash_table.DataTable(
        id="datatable-id",

        
        columns=table_columns,
        data=initial_records,
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=10,

        # Allow horizontal scrolling if the table is too wide
        style_table={
            "overflowX": "auto"
        },

        # Improve the appearance and readability of table cells
        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "minWidth": "120px",
            "maxWidth": "250px",
            "whiteSpace": "normal"
        },

        # Make the table headings easier to identify
        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    #breed chart and location map next to each other
    html.Div(
        style={
            "display": "flex",
            "gap": "20px",
            "flexWrap": "wrap"
        },

        children=[

            #display the breed chart
            html.Div(
                id="graph-id",
                style={
                    "flex": "1",
                    "minWidth": "500px"
                }
            ),

            #animal location map
            html.Div(
                id="map-id",
                style={
                    "flex": "1",
                    "minWidth": "500px"
                }
            )
        ]
    )
])

# Update the table when a filter is selected


@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("status-message", "children")
    ],
    Input("filter-type", "value")
)
def update_dashboard(filter_type):

    try:
        # search for the matching animal records
        records = service.get_animals(filter_type)

        # Create a message that tells the user how many records were found
        if records:
            message = "{} animal records found.".format(
                len(records)
            )
        else:
            message = (
                "No animals matched the selected rescue criteria."
            )

        return records, table_columns, message

    except ValueError:
        #message if an invalid filter is received
        return (
            [],
            table_columns,
            "The selected rescue filter is not valid."
        )

    except RuntimeError as error:
        # Print error for troubleshooting
        print(error)

        # Shows message without displaying database info
        return (
            [],
            table_columns,
            "The dashboard could not retrieve animal records."
        )



# Update the chart using the visible records


@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graph(view_data):
    #Create a chart showing the most common breeds

    # Use all starting records 
    if view_data is None:
        view_data = initial_records

    #count the most common breeds
    summary = service.get_breed_summary(view_data)

    #shows message if chart is empty
    if summary.empty:
        return html.P(
            "No breed information is available."
        )

    # Create a bar chart showing the ten most common breeds
    figure = px.bar(
        summary,
        x="breed",
        y="count",
        title="Top Breeds in Current Results",
        labels={
            "breed": "Breed",
            "count": "Number of Animals"
        }
    )

    # Angled names so you can actually read them
    figure.update_layout(
        xaxis_tickangle=-35
    )

    return dcc.Graph(
        figure=figure
    )



# Highlight a selected table column


@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),
    Input(
        "datatable-id",
        "selected_columns"
    )
)
def update_styles(selected_columns):
    #Highlight the column selected

    return [
        {
            "if": {
                "column_id": column
            },
            "backgroundColor": "##F2F2F2" #light gray
        }
        for column in selected_columns
    ]



# Update the map using the selected animal


@app.callback(
    Output("map-id", "children"),
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(view_data, selected_rows):
    #Displays the location of the selected animal

    # Use the starting records if the table data is not available
    if view_data is None:
        view_data = initial_records

    
    animal = service.get_selected_animal(
        view_data,
        selected_rows
    )

    # Display a message if the animal does not have a valid location
    if animal is None:
        return html.P(
            "No location is available for this animal."
        )

    # Create a map centered on the selected animal
    return dl.Map(
        style={
            "width": "100%",
            "height": "500px"
        },

        center=[
            animal["latitude"],
            animal["longitude"]
        ],

        zoom=10,

        children=[

            # Display the map background
            dl.TileLayer(),

            # Places a marker for location
            dl.Marker(
                position=[
                    animal["latitude"],
                    animal["longitude"]
                ],

                children=[

                    # Display the breed 
                    dl.Tooltip(
                        animal["breed"]
                    ),

                    # Display name and breed when clicked
                    dl.Popup([

                        html.H3(
                            "Animal Information"
                        ),

                        html.P(
                            "Name: {}".format(
                                animal["name"]
                            )
                        ),

                        html.P(
                            "Breed: {}".format(
                                animal["breed"]
                            )
                        )
                    ])
                ]
            )
        ]
    )



# Export to a CSV file


@app.callback(
    Output("download-csv", "data"),
    Input("export-button", "n_clicks"),
    State("datatable-id", "derived_virtual_data"),
    prevent_initial_call=True
)
def export_results(n_clicks, view_data):
    #Download the records displayed in the table

    if not n_clicks:
        return no_update

    # Use all starting records if the table wasnt changed
    if view_data is None:
        view_data = initial_records

    # doesnt create empty file
    if not view_data:
        return no_update

    try:
        csv_content = service.records_to_csv(
            view_data
        )

        return {
            "content": csv_content, "filename": "animal_shelter_results.csv", "type": "text/csv"
        }

    except ValueError:
        return no_update



# Run the dashboard


app.run(
    debug=False,
    port=8050,
    jupyter_mode="tab" #opens in a tab instead of cell
)

Connected to MongoDB
Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>